# Streamlit Assignment

**Created by Muhammad Farzam Khan**




## 1. Load Dataset in Jupyter notebook

In [ ]:
import pandas as pd
import numpy as np
import pickle

data = pd.read_csv('Employee(2).csv')
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


In [2]:
print('Shape:', data.shape)
print('Missing Values:')
print(data.isna().sum())
print('\nChurn distribution:')
print(data.LeaveOrNot.value_counts(normalize=True))

Shape: (4653, 9)
Missing Values:
Education                    0
JoiningYear                  0
City                         0
PaymentTier                  0
Age                          0
Gender                       0
EverBenched                  0
ExperienceInCurrentDomain    0
LeaveOrNot                   0
dtype: int64

Churn distribution:
LeaveOrNot
0    0.656136
1    0.343864
Name: proportion, dtype: float64


## 2. Select Features and Target

In [ ]:
# Splitting features and target variable
X = data.drop(columns=['LeaveOrNot'])
y = data['LeaveOrNot']


categorical_features = list(X.select_dtypes(include=[object]).columns) + ['PaymentTier']
numerical_features = list(X.select_dtypes(include=[np.float64, np.int64]).columns.difference(['PaymentTier']))

print('Categorical Features:', categorical_features)
print('Numerical Features:', numerical_features)

Categorical Features: ['Education', 'City', 'Gender', 'EverBenched', 'PaymentTier']
Numerical Features: ['Age', 'ExperienceInCurrentDomain', 'JoiningYear']


C:\Users\Farzam\AppData\Local\Temp\ipykernel_16380\4116951060.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = list(X.select_dtypes(include=[object]).columns) + ['PaymentTier']


## 3. Splitting Data for Train/Test

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)

Training shape: (3722, 8)
Testing shape: (931, 8)


## 4. Generate & Export Feature Input Metadata

In [10]:
def summarize_cat(data, categorical_features):
    results = []
    for column in data[categorical_features]:
        members = data[column].unique().tolist()
        results.append([column, members])
    return pd.DataFrame(results, columns=['Column Name', 'Members'])

summarize_cat(X_train, categorical_features)

,Column Name,Members
0,Education,"[Masters, Bachelors, PHD]"
1,City,"[New Delhi, Bangalore, Pune]"
2,Gender,"[Male, Female]"
3,EverBenched,"[No, Yes]"
4,PaymentTier,"[3, 2, 1]"


In [11]:
my_feature_dict = {
    'CATEGORICAL': summarize_cat(X_train, categorical_features).to_dict(),
    'NUMERICAL': {
        'Column Name': numerical_features,
        'Default': {col: int(X_train[col].median()) for col in numerical_features},
        'Min': {col: int(X_train[col].min()) for col in numerical_features},
        'Max': {col: int(X_train[col].max()) for col in numerical_features},
    }
}

my_feature_dict

{'CATEGORICAL': {'Column Name': {0: 'Education',
   1: 'City',
   2: 'Gender',
   3: 'EverBenched',
   4: 'PaymentTier'},
  'Members': {0: ['Masters', 'Bachelors', 'PHD'],
   1: ['New Delhi', 'Bangalore', 'Pune'],
   2: ['Male', 'Female'],
   3: ['No', 'Yes'],
   4: [3, 2, 1]}},
 'NUMERICAL': {'Column Name': ['Age',
   'ExperienceInCurrentDomain',
   'JoiningYear'],
  'Default': {'Age': 28, 'ExperienceInCurrentDomain': 3, 'JoiningYear': 2015},
  'Min': {'Age': 22, 'ExperienceInCurrentDomain': 0, 'JoiningYear': 2012},
  'Max': {'Age': 41, 'ExperienceInCurrentDomain': 7, 'JoiningYear': 2018}}}

In [12]:
with open('my_feature_dict.pkl', 'wb') as fp:
    pickle.dump(my_feature_dict, fp)

print('Schema exported successfully: my_feature_dict.pkl')

Schema exported successfully: my_feature_dict.pkl


## 5. Creating the Preprocessing & ML Pipeline

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# Numeric pipeline: handle missing numbers, then standardize
pipeline_num = Pipeline(steps=[
    ('simple_imputer1', SimpleImputer(strategy='median')),
    ('scale_data', StandardScaler()),
])

# Categorical pipeline: dummy/one-hot encoding
pipeline_cat = Pipeline(steps=[
    ('OneHotEncode', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_stage_2 = ColumnTransformer(
    transformers=[
        ('cat', pipeline_cat, categorical_features),
        ('num', pipeline_num, numerical_features),
    ], remainder='drop'
)

preprocessor_stack = Pipeline(steps=[
    ('preprocessor_stage_2', preprocessor_stage_2)
])

# Same RandomForestClassifier used in the class notebook
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_stack),
    ('classifier', RandomForestClassifier(random_state=42))
])

pipeline.fit(X_train, y_train)
print('Pipeline fitted successfully!')

Pipeline fitted successfully!


## 6. Evaluate the Model

In [14]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Training set
y_train_pred = pipeline.predict(X_train)
print('Training Accuracy:', accuracy_score(y_train, y_train_pred))

# Testing set
y_test_pred = pipeline.predict(X_test)
print('Testing Accuracy:', accuracy_score(y_test, y_test_pred))
print('\nClassification Report:\n', classification_report(y_test, y_test_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_test_pred))

Training Accuracy: 0.9279957012358947
Testing Accuracy: 0.8539205155746509

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.91      0.89       610
           1       0.82      0.74      0.78       321

    accuracy                           0.85       931
   macro avg       0.84      0.83      0.83       931
weighted avg       0.85      0.85      0.85       931


Confusion Matrix:
 [[558  52]
 [ 84 237]]


## 7. Export & Reload the Trained Pipeline

The lecture used `dill` because its Telco pipeline had a custom `lambda` transformation. This employee pipeline needs no custom lambda; **standard Python `pickle`** saves and loads it directly. The file name and Streamlit usage remain the same: `pipeline.pkl`.

In [15]:
# Export fitted ML pipeline
with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)
print('Pipeline exported successfully: pipeline.pkl')

# Reload the model and verify a prediction
with open('pipeline.pkl', 'rb') as file:
    loaded_pipeline = pickle.load(file)

my_pred_array = X_test.iloc[15:16]
print('Employee data:')
print(my_pred_array)
print('Prediction (0 = Stay, 1 = Leave):', loaded_pipeline.predict(my_pred_array))

Pipeline exported successfully: pipeline.pkl
Employee data:
      Education  JoiningYear  City  PaymentTier  Age  Gender EverBenched  \
1128  Bachelors         2015  Pune            2   26  Female          No   

      ExperienceInCurrentDomain  
1128                          4  
Prediction (0 = Stay, 1 = Leave): [1]


## Run Streamlit App

streamlit run "C:\Users\Farzam\Desktop\Karachi AI CDA Batch 12\ML\Employee_Churn_Streamlit_Assignment.app.py"